# Herramienta 02 - Clasificación de conducción distractiva

Este notebook funciona como una herramienta de clasificación visual: prepara una muestra de imágenes, extrae características, entrena un modelo, evalúa errores y permite clasificar una imagen nueva.


## 1. Configuración

El notebook descarga las imágenes preprocesadas desde el repositorio público de GitHub, entrena el clasificador y guarda el modelo entrenado como `modelo_conduccion.pkl`. Ese archivo se sube al repo para que la página web lo use directamente sin necesidad de reentrenar.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, subprocess
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

REPO = 'https://github.com/AndresGuido9820/sistema-transporte-inteligente.git'
BASE_URL = 'https://raw.githubusercontent.com/AndresGuido9820/sistema-transporte-inteligente/main'
IMG_DIR = Path('sistema-transporte-inteligente/data/processed/driver_images_real')

# Descarga solo la carpeta de imágenes usando sparse checkout (rápido, un solo clone)
if not IMG_DIR.exists():
    print('Descargando imágenes desde GitHub (sparse clone)...')
    subprocess.run(['git', 'clone', '--depth=1', '--filter=blob:none', '--sparse', REPO], check=True)
    subprocess.run(
        ['git', 'sparse-checkout', 'set',
         'data/processed/driver_images_real',
         'data/processed/driver_images.csv'],
        cwd='sistema-transporte-inteligente', check=True
    )
    print('Listo.')
else:
    print('Imágenes ya disponibles.')

LOCAL_DIR = IMG_DIR

## 2. Carga de metadatos e imágenes

**Fuente del dataset:** [Multi-Class Driver Behavior Image Dataset](https://zenodo.org/records/14908802) — publicado en Zenodo y Mendeley Data ([DOI: 10.17632/mzb4b6dff3.1](https://data.mendeley.com/datasets/mzb4b6dff3/1)). Asociado al artículo científico: *"A multi-class driver behavior dataset for real-time detection and road safety enhancement"* (PMC, 2025).

El dataset fue recolectado en Ashulia, Dhaka (Bangladesh) durante octubre de 2024 con cámaras de teléfono móvil en vehículos privados y buses públicos. Contiene 7,286 imágenes de alta resolución en 5 clases de comportamiento. El notebook usa 200 imágenes preprocesadas (40 por clase) almacenadas en el repositorio para garantizar reproducibilidad sin depender de descarga externa.

**Limitación conocida:** El dataset no incluye una clase explícita de somnolencia (`drowsiness`). Conductas como dormir al volante quedan dentro de `other_activities`.

In [ ]:
csv_path = Path('sistema-transporte-inteligente/data/processed/driver_images.csv')
metadata = pd.read_csv(csv_path)
metadata['local_path'] = metadata['image_path'].apply(
    lambda p: str(Path('sistema-transporte-inteligente') / p)
)

print('Total imágenes:', len(metadata))
display(metadata['label'].value_counts().rename_axis('clase').reset_index(name='imagenes'))

## 3. Exploración visual
Se muestran ejemplos por clase para comprobar que el problema es visualmente razonable y que las etiquetas tienen sentido.


In [ ]:
clases = sorted(metadata['label'].unique())
plt.figure(figsize=(12, 7))
plot_index = 1
for clase in clases:
    ejemplos = metadata[metadata['label'] == clase].head(3)
    for _, row in ejemplos.iterrows():
        plt.subplot(len(clases), 3, plot_index)
        plt.imshow(Image.open(row['local_path']).convert('RGB'))
        plt.title(clase, fontsize=9)
        plt.axis('off')
        plot_index += 1
plt.tight_layout()
plt.show()


## 4. Extracción de características

Se combinan características de color (medias RGB, desviación estándar, ratio oscuro/brillante) con **HOG** (*Histogram of Oriented Gradients*), un descriptor de textura y forma que captura bordes y patrones locales — clave para distinguir comportamientos como `texting_phone` vs `talking_phone`. El resultado es un vector de 136 características por imagen (8 de color + 128 de HOG).

In [ ]:
from skimage.feature import hog as hog_features

def extraer_caracteristicas(path):
    img_rgb  = Image.open(path).convert('RGB').resize((64, 64))
    img_gray = img_rgb.convert('L')
    arr      = np.asarray(img_rgb,  dtype=float) / 255.0
    arr_gray = np.asarray(img_gray, dtype=float) / 255.0

    # Color (8 valores)
    medias      = arr.mean(axis=(0, 1))
    desvios     = arr.std(axis=(0, 1))
    brillo      = arr.mean(axis=2)
    dark_ratio  = (brillo < 0.20).mean()
    bright_ratio= (brillo > 0.80).mean()
    color_feats = np.concatenate([medias, desvios, [dark_ratio, bright_ratio]])

    # HOG — 128 valores (8 orientaciones, celdas 16×16, bloques 1×1)
    hog_feats = hog_features(
        arr_gray,
        orientations=8,
        pixels_per_cell=(16, 16),
        cells_per_block=(1, 1),
        feature_vector=True,
    )
    return np.concatenate([color_feats, hog_feats])  # 136 total

X = np.vstack([extraer_caracteristicas(path) for path in metadata['local_path']])
y = metadata['label'].values
print('Matriz de entrenamiento:', X.shape)

## 5. Entrenamiento y métricas
Se entrena un clasificador multiclase y se reportan accuracy, precisión, recall y F1 por clase.


In [ ]:
X_train, X_test, y_train, y_test, path_train, path_test = train_test_split(
    X, y, metadata['local_path'].values,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

modelo = RandomForestClassifier(n_estimators=160, random_state=SEED, class_weight='balanced')
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45, cmap='Blues')
plt.title('Matriz de confusión')
plt.tight_layout()
plt.show()


## 6. Aciertos y errores
Esta sección ayuda a explicar en el reporte qué clases se confunden y por qué se necesitan más datos o una CNN para mejorar.


In [ ]:
revision = pd.DataFrame({'path': path_test, 'real': y_test, 'prediccion': pred})
revision['correcta'] = revision['real'] == revision['prediccion']
display(revision.groupby(['real', 'prediccion']).size().reset_index(name='casos').sort_values('casos', ascending=False).head(10))

muestras = pd.concat([revision[revision['correcta']].head(3), revision[~revision['correcta']].head(3)])
plt.figure(figsize=(12, 4))
for i, (_, row) in enumerate(muestras.iterrows(), start=1):
    plt.subplot(1, len(muestras), i)
    plt.imshow(Image.open(row['path']).convert('RGB'))
    plt.title(f"R: {row['real']}\nP: {row['prediccion']}", fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()


## 7. Herramienta de clasificación
Cambie `imagen_a_clasificar` por la ruta de una imagen subida a Colab. Si se deja vacío, se usa una imagen de prueba.


In [ ]:
#@title Parámetros de la herramienta
imagen_a_clasificar = '' #@param {type:'string'}

if not imagen_a_clasificar:
    imagen_a_clasificar = path_test[0]

features_img = extraer_caracteristicas(imagen_a_clasificar).reshape(1, -1)
clase = modelo.predict(features_img)[0]
probas = pd.Series(modelo.predict_proba(features_img)[0], index=modelo.classes_).sort_values(ascending=False)

plt.figure(figsize=(5, 4))
plt.imshow(Image.open(imagen_a_clasificar).convert('RGB'))
plt.title(f'Clase predicha: {clase}')
plt.axis('off')
plt.show()

display(probas.rename('probabilidad').reset_index().rename(columns={'index': 'clase'}))
probas.plot(kind='bar', figsize=(8, 3), title='Probabilidad por clase')
plt.ylabel('Probabilidad')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.25)
plt.show()


## 8. Informe: tipos de distracción y medidas preventivas

### 8.1 Distribución de comportamientos en el dataset

El dataset está balanceado con **40 imágenes por clase** (200 en total). Esto significa que no hay una clase más "frecuente" por diseño, pero los resultados del modelo revelan cuáles comportamientos son más difíciles de detectar en la práctica — lo que tiene implicaciones directas sobre el riesgo operativo.

| Comportamiento | F1-score | Riesgo operativo | Confusión principal |
|---|---|---|---|
| `texting_phone` | 0.70 | Muy alto | Se confunde con `talking_phone` |
| `talking_phone` | 0.53 | Alto | Se confunde con `other_activities` |
| `safe_driving` | 0.56 | Medio (falsos positivos) | Se confunde con `talking_phone` |
| `turning` | 0.78 | Bajo | Visualmente distintivo |
| `other_activities` | 0.80 | Variable | Absorbe casos ambiguos |

### 8.2 Distracciones más críticas identificadas

**1. Uso del teléfono (texting + talking — 80 imágenes del dataset)**
Son las dos clases con mayor riesgo. `texting_phone` implica mirada desviada hacia la pantalla y manos ocupadas; `talking_phone` ocupa al menos una mano. Juntas representan la mayor fuente de accidentes en transporte motorizado según la OMS. El modelo las confunde entre sí en 4 casos del conjunto de prueba, lo que indica que visualmente comparten patrones similares (mano cerca de la cara).

**2. Otras actividades distractoras (other_activities)**
Esta clase agrupa conductas heterogéneas: comer, beber, hablar con pasajeros y —limitación del dataset— somnolencia. El modelo la predice con recall perfecto (1.00) pero la usa como "cajón de sastre" para casos inciertos, generando falsos positivos. En operación real, cualquier alerta de esta clase requeriría validación humana.

**3. Giro / maniobras (turning)**
Aunque es la clase más fácil de detectar (precisión 0.88), en transporte público los giros bruscos sin atención adecuada son un factor de riesgo para pasajeros de pie.

### 8.3 Medidas preventivas por tipo de distracción

**Uso del teléfono móvil:**
- Instalar bloqueadores de pantalla automáticos cuando el vehículo supera 20 km/h
- Capacitación obligatoria semestral con evidencia de accidentes reales
- Política de teléfono en modo silencio y guardado antes de arrancar
- Alerta automática al supervisor ante detección sostenida (>3 segundos) de la clase

**Otras actividades distractoras:**
- Revisar condiciones de trabajo: turnos prolongados y fatiga son la causa subyacente más común
- Implementar pausas activas obligatorias cada 2 horas en rutas largas
- Integrar cámara con detección de cierre de ojos (dataset DDD o MRL) para somnolencia específica
- Prohibición de consumo de alimentos durante la conducción

**Conducción en maniobras:**
- Entrenamiento en técnicas de giro y cambio de carril con espejos
- Revisión periódica de puntos ciegos en cada unidad

### 8.4 Limitaciones del sistema para uso en producción

- El modelo actual es un prototipo de clasificación por fotograma individual. Un sistema real requiere análisis de secuencias de video (al menos 3–5 segundos continuos) para evitar falsas alertas por posturas momentáneas.
- La ausencia de detección facial explícita (puntos clave de ojos, boca) limita la detección de somnolencia, que es el comportamiento de mayor riesgo en transporte de pasajeros.
- Ninguna alerta del modelo debe traducirse en sanción automática sin revisión humana.

## 9. Conclusión

El clasificador alcanzó una **exactitud del 68%** sobre el conjunto de prueba (50 imágenes, 10 por clase), lo que representa 3.4 veces mejor que la clasificación aleatoria (20% con 5 clases equiprobables). Este resultado es aceptable considerando que el modelo usa únicamente características manuales — HOG y estadísticas de color — sobre imágenes redimensionadas a 64×64 píxeles, sin recurrir a redes neuronales convolucionales.

**Clases con mejor desempeño:**
`turning` obtuvo la mayor precisión (0.88) porque el giro del volante genera posturas corporales visualmente distintas y fácilmente capturables por el descriptor HOG. `other_activities` logró recall perfecto (1.00), pero con una precisión menor (0.67): el modelo tiende a clasificar en esta categoría los casos ambiguos, lo que infla su sensibilidad a costa de generar falsos positivos.

**Clases más problemáticas:**
`talking_phone` y `safe_driving` presentan los F1-scores más bajos (0.53 y 0.56 respectivamente). La causa principal es la similitud visual entre ambas: en muchos encuadres, la posición de la mano y la cabeza de un conductor hablando por teléfono es difícil de distinguir de la conducción normal usando solo características de bajo nivel. Esta confusión se confirma en la tabla de errores, donde `talking_phone → other_activities` (3 casos) y `safe_driving → talking_phone` (2 casos) son los errores más frecuentes.

**Limitaciones del enfoque actual:**
El uso de 200 imágenes para entrenar 5 clases es insuficiente para generalizar en escenarios reales. Las características HOG capturan textura y bordes pero no semántica de alto nivel como la posición del teléfono, la dirección de la mirada o el estado de los ojos. Adicionalmente, el dataset no incluye una clase explícita de somnolencia: los comportamientos de sueño al volante quedan agrupados dentro de `other_activities`, lo cual limita la aplicabilidad del sistema en entornos de transporte público donde la fatiga es un factor crítico.

**Vías de mejora concretas:**
Para producción, el paso natural es reemplazar el extractor de características manual por una CNN preentrenada como MobileNetV3 o EfficientNet-B0 mediante transfer learning, lo que históricamente eleva la exactitud en este tipo de problemas al rango 88–95%. Paralelamente, incorporar un módulo dedicado de detección de somnolencia basado en el estado de los ojos (datasets MRL o DDD) permitiría cubrir el comportamiento de riesgo más relevante para la operación de flotas de transporte público.

**Impacto operativo:**
A pesar de sus limitaciones actuales, el sistema demuestra que es viable automatizar la detección de conducción distractiva con herramientas de visión por computador accesibles y sin infraestructura de servidor. Integrado con una cámara a bordo y un umbral de alerta configurado por clase (priorizando `texting_phone` y `talking_phone` por su mayor riesgo), este módulo puede convertirse en una capa preventiva real dentro de un sistema de monitoreo de flota, complementando la supervisión humana sin reemplazarla.